# 04: Evaluation practice (teardown T4–T7)

Predictions were committed before the runs (`analysis/PREDICTIONS.md`); deviations are in
`analysis/DEVIATIONS.md`.

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from heart_audit.conventional import encode, split_indices
from heart_audit.data import PROJECT_ROOT, load_kaggle918
from heart_audit.evaluation import site_weighted_auc_ci, auc
from heart_audit.plots import effect_summary, paired_difference, site_dumbbell
from heart_audit.seeds import derive

R = json.loads((PROJECT_ROOT / "results" / "teardown.json").read_text(encoding="utf-8"))
T = PROJECT_ROOT / "results" / "teardown"
control = pd.read_csv(PROJECT_ROOT / "results" / "control_seeds.csv")
def verdict_table(result, keys):
    rows = []
    for k in keys:
        v = result[k]
        if isinstance(v, list):
            rows += [{"prediction": f"{k} ({g['column']})", "verdict": g["verdict"]} for g in v]
        else:
            rows.append({"prediction": k, "verdict": v["verdict"]})
    return pd.DataFrame(rows)

## T4: One split cannot rank the models

In [2]:
pd.Series(R["P4.1"]["win_share"], name="share of seeds where the model is most accurate (ties split)").to_frame()

,share of seeds where the model is most accurate (ties split)
decision_tree,0.0003
logistic_regression,0.2187
random_forest,0.4960
svm,0.2850


In [3]:
d = control.random_forest - control.logistic_regression
paired_difference(d, "random forest minus logistic regression, test-set accuracy on the same split",
                  "Which model 'wins' depends on the split", PROJECT_ROOT / "images" / "rf_vs_lr_by_seed.png")
{k: R[k] for k in ("P4.2", "P4.3")}

{'P4.2': {'sd_rf_minus_lr': 0.017388100271448673,
  'sd_rf': 0.02256302003701568,
  'mean_rf_minus_lr': 0.008168478260869562,
  'share_rf_ahead': 0.632,
  'share_lr_ahead': 0.265,
  'verdict': 'supported'},
 'P4.3': {'share_significant': 0.03, 'verdict': 'supported'}}

![](../images/rf_vs_lr_by_seed.png)

Random forest averages 0.8 points above logistic regression, but it is ahead on only 63% of
splits. An exact McNemar test finds a significant difference on 3% of splits. The paired
difference varies less than either model's accuracy (P4.2): the models are wrong on many of
the same patients, so comparing them on one split is more stable than their raw accuracies
suggest. It is still not stable enough to rank them.

## T5: Reporting the best of several models

In [4]:
bias = pd.read_csv(T / "t5_bias.csv").bias * 100
print(f"winner's-curse bias: mean {bias.mean():.2f} pp, central 95% [{bias.quantile(0.025):.2f}, {bias.quantile(0.975):.2f}]")
R["P5.1"]

winner's-curse bias: mean 1.48 pp, central 95% [0.28, 2.52]


{'mean_bias_pp': 1.4813832566829284,
 'central95_pp': [0.282976006431791, 2.523920331730849],
 'naive_best_of_4_median': 0.875,
 'verdict': 'supported'}

This applies to the 10 of 30 surveyed notebooks that report the best of several models on
one test set. It is not the modal practice.

## T6: Fitting transforms before the split

In [5]:
t6 = pd.read_csv(T / "t6_seeds.csv")
pd.DataFrame({m: {"median full - train (pp)": 100 * t6[f"scale_{m}"].median(), "mean |diff| (pp)": 100 * t6[f"scale_{m}"].abs().mean()}
              for m in ["decision_tree", "logistic_regression", "random_forest", "svm"]}).T.round(3)

,median full - train (pp),mean |diff| (pp)
decision_tree,0.0,0.245
logistic_regression,0.0,0.007
random_forest,0.0,0.180
svm,0.0,0.098


In [6]:
{k: R[k] for k in ("T6_sanity", "P6.2", "T6_xgboost")}

{'T6_sanity': {'prediction_identity': {'decision_tree': 0.996538043478261,
   'logistic_regression': 0.9999347826086957,
   'random_forest': 0.9978478260869567,
   'svm': 0.9988967391304348},
  'passed': False},
 'P6.2': {'median_paired_diff': 0.0,
  'central95': [-0.010869565217391353, 0.010869565217391353],
  'verdict': 'supported'},
 'T6_xgboost': {'native': 0.8586956521739131,
  'full': 0.8532608695652174,
  'train': 0.8532608695652174,
  'median_full_minus_train': 0.0,
  'median_native_minus_train': 0.005434782608695565}}

The pre-registered sanity check (identical tree predictions on ≥ 99.9% of rows) fails narrowly.
The next cell shows why: an exactly representable rescaling changes no tree prediction, and a
standardisation changes a few. Floating-point rounding flips tied splits (the one-hot encoding
has complementary columns). No test-row information is involved.

In [7]:
X, y = encode(load_kaggle918()); X = X.to_numpy()
changed_std = changed_exact = total = 0
for seed in derive("control", 200):
    tr, te = split_indices(len(X), seed)
    base = DecisionTreeClassifier(random_state=seed).fit(X[tr], y[tr]).predict(X[te])
    sd_full, sd_tr = X.std(0), X[tr].std(0); sd_full[sd_full == 0] = 1; sd_tr[sd_tr == 0] = 1
    A, B = (X - X.mean(0)) / sd_full, (X - X[tr].mean(0)) / sd_tr
    pa = DecisionTreeClassifier(random_state=seed).fit(A[tr], y[tr]).predict(A[te])
    pb = DecisionTreeClassifier(random_state=seed).fit(B[tr], y[tr]).predict(B[te])
    p4 = DecisionTreeClassifier(random_state=seed).fit(4 * X[tr], y[tr]).predict(4 * X[te])
    changed_std += (pa != pb).sum(); changed_exact += (p4 != base).sum(); total += len(te)
print(f"standardised on all rows vs training rows: {changed_std}/{total} predictions differ")
print(f"unscaled vs x4 (exact in floating point):  {changed_exact}/{total}")

standardised on all rows vs training rows: 128/36800 predictions differ
unscaled vs x4 (exact in floating point):  0/36800


On this dataset, the leak that textbooks warn about most, fitting the scaler before the split,
changes nothing measurable. Median-imputing `Cholesterol == 0` on all rows does not either.
Not every leak matters. The one that does (T1) is in the data, not the code.

## T7: Random folds vs held-out hospitals

In [8]:
oof = pd.read_csv(T / "t7_oof.csv")
rows = []
for m in ["logistic_regression", "random_forest"]:
    r = R["P7.1"][m]
    wl = site_weighted_auc_ci(oof.y, oof[f"{m}_loso"], oof.source, 2000, seed=1)
    wk = site_weighted_auc_ci(oof.y, oof[f"{m}_kfold"], oof.source, 2000, seed=1)
    rows.append({"model": m, "pooled AUC, LOSO (biased low under the null)": r["auc_loso"], "pooled AUC, 10-fold": r["auc_kfold"],
                 "pooled difference [95% CI]": f"{r['diff']:+.3f} [{r['ci95'][0]:+.3f}, {r['ci95'][1]:+.3f}]",
                 "within-site AUC, LOSO": wl[0], "within-site AUC, 10-fold": wk[0]})
pd.DataFrame(rows).set_index("model").round(3)

,"pooled AUC, LOSO (biased low under the null)","pooled AUC, 10-fold",pooled difference [95% CI],"within-site AUC, LOSO","within-site AUC, 10-fold"
model,,,,,
logistic_regression,0.834,0.875,"-0.040 [-0.050, -0.031]",0.830,0.838
random_forest,0.814,0.861,"-0.048 [-0.063, -0.032]",0.809,0.814


In [9]:
r = R["P7.1"]["logistic_regression"]["per_site"]
sites = list(r)
site_dumbbell(sites, [r[s]["loso"] for s in sites], [r[s]["kfold"] for s in sites],
              "hospital held out", "random 10-fold", "Logistic regression AUC within each hospital",
              PROJECT_ROOT / "images" / "t7_sites.png",
              notes=[f"{r[s]['n_pos']} diseased / {r[s]['n_neg']} healthy" for s in sites])

WindowsPath('C:/Users/ethan/Desktop/Coding Projects/heart-disease-audit/images/t7_sites.png')

![](../images/t7_sites.png)

P7.1 is supported as registered: pooled AUC drops by about 4 points when a whole hospital is
held out. Two caveats:
- Deviation D6: pooled LOSO AUC is biased low by about 1 point even with no signal.
- Within each hospital, the two schemes differ much less. Most of the pooled gap is
  between-hospital, not a loss of ranking ability within a hospital.

Switzerland has 8 healthy patients, so its AUC rests on 8 negatives and is flagged wherever it
appears.

## What each practice is worth

In [10]:
t1 = pd.read_csv(T / "t1_seeds.csv"); t3 = pd.read_csv(T / "t3_seeds.csv")
def row(label, values):
    v = 100 * np.asarray(values)
    return {"label": label, "median": float(np.median(v)), "lo": float(np.percentile(v, 2.5)), "hi": float(np.percentile(v, 97.5))}
rows = [
    row("Label-decided filled values\n(in the published CSV)", t1.published - t1.reverted),
    row("Duplicates kept (1,190-row merge;\nnot in the published CSV)", t3.random - t3.grouped),
    row("Best of 4 models reported\n(10 of 30 notebooks)", pd.read_csv(T / "t5_bias.csv").bias),
    row("Scaler fit before the split\n(19 of 30 notebooks fit something)", t6.scale_random_forest),
    row("Cholesterol = 0 imputed\nbefore the split", t6.impute_rf),
]
effect_summary(rows, PROJECT_ROOT / "images" / "effect_summary.png")
pd.DataFrame(rows).set_index("label").round(2)

,median,lo,hi
label,,,
Label-decided filled values\n(in the published CSV),7.07,2.17,12.50
"Duplicates kept (1,190-row merge;\nnot in the published CSV)",7.90,1.71,14.48
Best of 4 models reported\n(10 of 30 notebooks),1.53,0.28,2.52
Scaler fit before the split\n(19 of 30 notebooks fit something),0.00,-0.54,0.54
Cholesterol = 0 imputed\nbefore the split,0.00,-1.09,1.09


![](../images/effect_summary.png)

## Verdicts

In [11]:
verdict_table(R, ['P4.1', 'P4.2', 'P4.3', 'P5.1', 'P6.1', 'P6.2', 'P7.1'])

,prediction,verdict
0,P4.1,supported
1,P4.2,supported
2,P4.3,supported
3,P5.1,supported
4,P6.1,supported
5,P6.2,supported
6,P7.1,supported
